In [1]:
import pandas as pd
import os
import pyterrier as pt
from ir_measures import *
import matplotlib.pyplot as plt
import seaborn as sns

if not pt.started():
    pt.init()

/tmp/ipykernel_2618/836136444.py:8: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_2618/836136444.py:9: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [2]:
docs = pd.read_pickle("/nfs/primary/sas_reranker/ohsumed_docs_w_t5base_sensitivity.pkl")
queries = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/queries.pkl")
qrels = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/qrels.pkl")

In [3]:
directory = "./runs_v2/"

baselines = []
baseline_names = []

sans = []
sans_names = []


for file in os.listdir(directory):
    if "splade" in file and "baseline" in file:
        baselines.append(pt.io.read_results(f"{directory}{file}"))
        baseline_names.append(file)
    if "splade" in file and "strategy" in file and "frozen" not in file:
        sans.append(pt.io.read_results(f"{directory}{file}"))
        sans_names.append(file)

In [4]:
def _sens_docs(qrels, run):
    if "sensitivity" in run.columns:
        run = run.drop(columns = ["sensitivity"])
    merged = pd.merge(run, docs, left_on = "doc_id", right_on = "docno")
    return merged.sensitivity.sum()
    
import ir_measures
sens_docs = ir_measures.define_byquery(
    _sens_docs, 
    name="sens_docs")

In [5]:
baseline_results = pt.Experiment(
    baselines,
    queries,
    qrels,
    eval_metrics=[nDCG@10, sens_docs@10],
    names = baseline_names
)

sans_results = pt.Experiment(
    sans,
    queries,
    qrels,
    eval_metrics=[nDCG@10, sens_docs@10],
    names = sans_names
)

combined_results = pt.Experiment(
    [pt.io.read_results(f"{directory}splade_combined_v1")],
    queries,
    qrels,
    eval_metrics=[nDCG@10, sens_docs@10],
    names = ["Combined V1"]
)

In [6]:
combined_results

,name,nDCG@10,sens_docs@10
0,Combined V1,0.407353,0.509434


In [10]:
# Extracting strategies from the names in sans_results
sans_results['strategy'] = sans_results['name'].apply(lambda x: x.split('_')[2])

# Assigning colors based on strategy using a color palette
palette = sns.color_palette("tab10", len(sans_results['strategy'].unique()))
strategy_colors = dict(zip(sans_results['strategy'].unique(), palette))

# Adding a color for the baseline strategy
strategy_colors["baseline"] = "black"
strategy_colors["combined"] = "purple"

# Map colors to sans_results
sans_results['color'] = sans_results['strategy'].map(strategy_colors)

# Assign baseline strategy and color
baseline_results['strategy'] = "baseline"
baseline_results['color'] = "black"

combined_results['strategy'] = "combined"
combined_results['color'] = "purple"

# Combine sans_results and baseline_results
combined_df = pd.concat([sans_results, baseline_results, combined_results], ignore_index=True)

# Plotting
plt.figure(figsize=(8, 6), dpi = 1080)
for strategy in combined_df['strategy'].unique():
    strategy_df = combined_df[combined_df['strategy'] == strategy]
    plt.scatter(
        strategy_df["sens_docs@10"], strategy_df["nDCG@10"],
        color=strategy_colors[strategy], s=100, label=strategy
    )

for i, row in combined_df.iterrows():
    if "baseline" in row["name"]:
        plt.text(row["sens_docs@10"] + 0.01, row["nDCG@10"], row["name"].split("_baseline")[0], fontsize=8, alpha=0.7)

plt.title("nDCG vs. sens_docs | SPLADE")
plt.xlabel("sens_docs@10")
plt.ylabel("nDCG@10")
plt.legend(title="Strategy")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("./plots/combined_v1.png", dpi = 1080)